# **Proyecto 02: Sistema de Clasificación de Especies de Aves a partir de Atributos Descriptivos**

---
## **Parte 1: Carga del dataset y preprocesado de los datos**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
import joblib
import os

### 1.1 Carga de `image_attribute_labels.txt`

El fichero tiene 5 columnas: `imgid`, `attrid`, `is_present`, `certainty_id`, `time`.  
Leemos solo las 3 primeras con `usecols=[0,1,2]` y les asignamos nombre.

In [ ]:
DATASET_PATH = 'CUB_200_2011'

# Carga del fichero de atributos (solo las 3 primeras columnas)
df_attr = pd.read_csv(
    os.path.join(DATASET_PATH, 'attributes', 'image_attribute_labels.txt'),
    sep='\s+',
    header=None,
    usecols=[0, 1, 2],
    names=['imgid', 'attrid', 'is_present']
)

print(f'Filas cargadas: {len(df_attr):,}')
print(f'Imágenes únicas: {df_attr["imgid"].nunique()}')
print(f'Atributos únicos: {df_attr["attrid"].nunique()}')
df_attr.head()

### 1.2 Reorganización: pivot por imagen

Cada imagen debe quedar en una fila con 312 columnas (una por atributo, valor 0/1).

In [ ]:
# Pivot: una fila por imgid, una columna por attrid
df_pivot = df_attr.pivot(index='imgid', columns='attrid', values='is_present')

print(f'Forma del DataFrame pivotado: {df_pivot.shape}')
print('(filas=imágenes, columnas=atributos)')
df_pivot.head()

### 1.3 Carga de etiquetas de clase (`image_class_labels.txt`)

In [ ]:
# Carga etiquetas verdaderas
df_labels = pd.read_csv(
    os.path.join(DATASET_PATH, 'image_class_labels.txt'),
    sep=' ',
    header=None,
    names=['imgid', 'class_id']
)
df_labels = df_labels.set_index('imgid')

print(f'Total etiquetas: {len(df_labels)}')
print(f'Clases únicas: {df_labels["class_id"].nunique()}')
df_labels.head()

### 1.4 Join atributos + etiquetas, barajar y dividir en train/test

In [ ]:
# Unir atributos con etiquetas
df_full = df_pivot.join(df_labels)

# Barajar para evitar patrones por orden
df_full = df_full.sample(frac=1, random_state=0)

# Separar atributos (X) de etiquetas (y)
X = df_full.drop(columns=['class_id'])
y = df_full['class_id']

# División 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print(f'Train: {X_train.shape[0]} imágenes')
print(f'Test:  {X_test.shape[0]} imágenes')
print(f'Atributos: {X_train.shape[1]}')

---
## **Parte 2: Entrenamiento y evaluación del modelo**

### 2.1 Random Forest con configuración inicial

Parámetros iniciales: `max_features=50`, `random_state=0`, `n_estimators=100`

In [ ]:
# Entrenamiento del Random Forest
rf = RandomForestClassifier(max_features=50, random_state=0, n_estimators=100, n_jobs=-1)
rf.fit(X_train, y_train)

# Evaluación
acc_train = rf.score(X_train, y_train)
acc_test  = rf.score(X_test, y_test)

print(f'Precisión en TRAIN: {acc_train:.4f} ({acc_train*100:.2f}%)')
print(f'Precisión en TEST:  {acc_test:.4f} ({acc_test*100:.2f}%)')

### 2.2 Matriz de confusión

Con 200 clases la matriz es muy grande (200×200). La mostramos como imagen de mapa de calor.

In [ ]:
# Carga de nombres de clases
df_classes = pd.read_csv(
    os.path.join(DATASET_PATH, 'classes.txt'),
    sep=' ',
    header=None,
    names=['class_id', 'class_name']
)
class_names = df_classes['class_name'].tolist()
print(f'Total clases: {len(class_names)}')
print('Ejemplo:', class_names[:5])

In [ ]:
# Predicciones en test
y_pred = rf.predict(X_test)

# Calcular matriz de confusión
cm = confusion_matrix(y_test, y_pred)

# Visualización como mapa de calor (200x200 es demasiado para ConfusionMatrixDisplay)
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.set_title(f'Matriz de Confusión (test)\nAccuracy: {acc_test*100:.2f}%', fontsize=14, pad=15)
plt.colorbar(im, ax=ax)
ax.set_xlabel('Clase predicha', fontsize=12)
ax.set_ylabel('Clase real', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()
print('Matriz guardada en confusion_matrix.png')

### 2.3 Búsqueda de hiperparámetros

Probamos distintas combinaciones de `max_features` y `n_estimators` para ver su efecto en la precisión.

In [ ]:
# Combinaciones de hiperparámetros a explorar
max_features_list = [10, 30, 50, 100, 150, 200]
n_estimators_list = [50, 100, 200]

resultados = []

for n_est in n_estimators_list:
    for max_feat in max_features_list:
        clf = RandomForestClassifier(
            max_features=max_feat,
            n_estimators=n_est,
            random_state=0,
            n_jobs=-1
        )
        clf.fit(X_train, y_train)
        acc = clf.score(X_test, y_test)
        resultados.append({'max_features': max_feat, 'n_estimators': n_est, 'accuracy': acc})
        print(f'max_features={max_feat:3d}, n_estimators={n_est:3d} → accuracy={acc:.4f}')

df_resultados = pd.DataFrame(resultados)
print('\nMejor combinación:')
print(df_resultados.loc[df_resultados['accuracy'].idxmax()])

In [ ]:
# Scatter plot: accuracy vs max_features, coloreado por n_estimators
fig, ax = plt.subplots(figsize=(10, 6))

colores = {50: 'blue', 100: 'orange', 200: 'green'}
for n_est in n_estimators_list:
    subset = df_resultados[df_resultados['n_estimators'] == n_est]
    ax.scatter(
        subset['max_features'],
        subset['accuracy'],
        color=colores[n_est],
        label=f'n_estimators={n_est}',
        s=100,
        zorder=5
    )
    ax.plot(subset['max_features'], subset['accuracy'], color=colores[n_est], alpha=0.5)

ax.set_xlabel('max_features (nº máximo de atributos por split)', fontsize=12)
ax.set_ylabel('Accuracy en test', fontsize=12)
ax.set_title('Evolución de la precisión según hiperparámetros del Random Forest', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('hyperparameter_search.png', dpi=100, bbox_inches='tight')
plt.show()
print('Gráfico guardado en hyperparameter_search.png')

### 2.4 Guardar el mejor modelo para la aplicación Streamlit

In [ ]:
# Seleccionar la mejor combinación de hiperparámetros
mejor = df_resultados.loc[df_resultados['accuracy'].idxmax()]
print(f'Mejor modelo: max_features={int(mejor.max_features)}, n_estimators={int(mejor.n_estimators)}')
print(f'Accuracy: {mejor.accuracy:.4f} ({mejor.accuracy*100:.2f}%)')

# Reentrenar con la mejor configuración sobre todos los datos de train
best_rf = RandomForestClassifier(
    max_features=int(mejor.max_features),
    n_estimators=int(mejor.n_estimators),
    random_state=0,
    n_jobs=-1
)
best_rf.fit(X_train, y_train)

# Guardar modelo y lista de columnas (necesarios para la app)
os.makedirs('models', exist_ok=True)
joblib.dump(best_rf, 'models/bird_rf_model.joblib')
joblib.dump(list(X_train.columns), 'models/feature_columns.joblib')

print('Modelo guardado en models/bird_rf_model.joblib')
print('Columnas guardadas en models/feature_columns.joblib')